In [1]:
# Imports and globals
%load_ext autoreload
%autoreload 2

import numpy as np
import jax.numpy as jnp
from scipy.linalg import sqrtm
from matplotlib import pyplot as plt
from ellipsoids import plot_ellipse, Ellipsoid
from typing import Tuple
from CompassGaitWalker import CompassGaitWalker
from diffrax import diffeqsolve, ODETerm, Tsit5, Event
import optimistix as optx
import jax
import pdb

PLOT = False


In [2]:
## Setup system and ODE Solver
sys = CompassGaitWalker(m=5., mh=10., a=0.5, b=0.5, gamma=0.0525) # This has a known limit cycle
x0_nom = np.array([-0.21876908,  0.32376908,  1.8083619 ,  1.49571536]) # This point must be on the guard
assert jnp.abs(sys.guard(0, x0_nom)) < 1e-5 # x0_nom is on guard

term = ODETerm(sys.f)
solver = Tsit5()
ctrl_func = lambda t, x: 0.0

cond_fn = sys.guard
root_finder = optx.Newton(1e-5, 1e-5, optx.rms_norm)
event = Event(cond_fn, root_finder, direction=True)

# Step-to-step map of the Compass Gait Walker
# Start on pre-impact state xminus and check next intersection onto guard surface
@jax.jit
def one_step(xm):
    xp = sys.reset(0, xm)
    traj = diffeqsolve(term, solver, t0=0, t1=1, dt0=0.001, y0=xp, args=ctrl_func, event=event)
    ys = traj.ys

    return ys.flatten()

print(x0_nom)
print(one_step(x0_nom))
print("Test one_step(x0_nom)")
assert jnp.allclose(one_step(x0_nom), x0_nom, atol=5e-4)
assert jnp.abs(sys.guard(0, x0_nom)) < 1e-5 # x0_nom is on guard

P = jax.vmap(one_step, 0) # Create vmap function over rows of the argument TODO: switch to map over columns

print("Test vmapped P")

assert jnp.allclose(P(x0_nom[np.newaxis, :]), x0_nom, atol=5e-4)

[-0.21876908  0.32376908  1.8083619   1.49571536]
[-0.21876908  0.32376908  1.8083619   1.49571536]
JitTracer<float32[4]>
JitTracer<float32[4]>
JitTracer<float32[4]>
JitTracer<float32[4]>
JitTracer<float32[4]>
[-0.21877202  0.32377207  1.8080168   1.4957036 ]
Test one_step(x0_nom)
[-0.21876908  0.32376908  1.8083619   1.49571536]
Test vmapped P


In [ ]:
## Functions for sampling on guard surface

N = 100 # Number of samples

# Define 3 vectors that are aligned with the guard surface so we can sample preimpact states
# You can do this automatically with jax.grad or by hand as shown here
guard_surface_chart = jnp.array([[1, -1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]]).T

# Randomly sample points uniformly from some rectangular interval in guard surface coordinates
guard_pts = np.random.uniform(-0.001, 0.001, size=(3, N))

# guard_pts: (3, N)
def guard_pts_to_full(guard_pts):
    return guard_surface_chart@guard_pts + x0_nom[:, np.newaxis]

def full_pts_to_guard(full_pts):
    return jnp.linalg.pinv(guard_surface_chart)@(full_pts - x0_nom[:, np.newaxis])

guard_vmap = jax.vmap(lambda x: sys.guard(0, x.flatten()), 0)
# print(guard_pts_to_full(perturbations).T)

assert guard_pts_to_full(guard_pts).shape == (4, N)
assert jnp.allclose(guard_vmap(guard_pts_to_full(guard_pts).T), 0, atol=1e-5)

In [31]:
## Algorithm subroutines for working with Ellipsoids

def sample_from(e: Ellipsoid, N_pts: int) -> np.ndarray:
    # Sample Normal Distribution 
    # TODO: pull dimension of ellipsoid
    d = 3
    pts = np.random.standard_normal((d, N_pts))
    r = np.random.uniform(0, 1, size=(N,)) ** (1/d) # The denominator of the power is also the dimension
    pts = r*pts/np.linalg.vector_norm(pts, axis=0)
    return np.linalg.inv(sqrtm(e.Q))@pts + e.c
    

def reject_and_resample(Ei: Ellipsoid, I, Pxi, rescale=0.9) -> Tuple[np.ndarray, Ellipsoid, np.ndarray]:
    N = Pxi.shape[1]
    in_pts = I[:, Ei.contains_point(Pxi)]
    Ein = Ellipsoid(4)
    Ein.fit_p_ball(in_pts.T, 2)
    Ein.scale(rescale) # Enforce contraction rate
    return sample_from(Ein, N), Ein, in_pts

In [ ]:
# Test Sampling from Ellipsoid 

E0 = Ellipsoid(3)

# Test sample_from
fig, ax = plt.subplots()
sample_pts = sample_from(E0, N)
ax.scatter(sample_pts[0, :], sample_pts[1, :])
plt.show()

# Test reject_and_resample
fig, ax = plt.subplots()
# Pxi = P(Ii.T).T

# P(in_pts) should lie completely within E0
# We want E_resample to tightly wrap in_pts
# P(I_resample) should also lie within E0
I_resample, E_resample, in_pts = reject_and_resample(E0, Ii, Pxi, rescale=1)

Pin_points = P(in_pts.T).T


Px_resample = P(I_resample.T).T
ax.scatter(Ii[0, :], Ii[1, :], c='black', marker='o')
ax.scatter(Pxi[0, :], Pxi[1, :], c='black', marker='x')
ax.scatter(I_resample[0, :], I_resample[1, :], c='green', marker='o')
ax.scatter(Px_resample[0, :], Px_resample[1, :], c='green', marker='x')
ax.scatter(in_pts[0, :], in_pts[1, :], c='blue', marker='o')
plot_ellipse(E0.Q, E0.c.flatten(), 1, ax, color = 'blue')
plot_ellipse(E_resample.Q, E_resample.c.flatten(), 1, ax, color = 'green')
plt.show()
